# Title: HRS Silver CDM DDL/DML Functional Specification — TEMPLATE
________________________________________
## 1. Document Information
|Document Name:|HRS Silver CDM DDL/DML Functional Specification|
| --- | --- |
|Version:|2.0 (Template)|
|Author:|Perez|
|Last Updated:|2026-08-02|
|Target Runtime:|Databricks Runtime 15.x|
|SQL Dialect:|Spark SQL|
|Storage Format:|Delta Lake|
|Deployment Environment:|Development|
|AI Assistant:|Claude|

**Changelog from v1.0:** Fixed §6 CATEG type contradiction with §12. Added §12a
(Column Variance Declaration). Added §13a (DML Generation Requirements — previously
undocumented and inherited only by convention). Expanded §4 Source Profile to
capture source shape, naming convention, and sample-coverage caveats so the
template holds for non-core-file sections (e.g., Leave-Behind).
________________________________________

## 2. Objective

Use Claude to generate Databricks SQL DDL and DML scripts using Spark SQL compatible
with Databricks Runtime 15.x.

The generated SQL creates one Silver CDM table for a single RAND HRS survey section.

Return only SQL.

## 3. Purpose

This specification defines the physical implementation of one Silver CDM
survey-section table, and the load logic that populates it.

|Area | Description |
| --- | --- |
|Table Structure | Physical table definition|
|Relationships | Parent-child relationships|
|Column Definitions | Data types and nullability|
|Business Rules | Required ETL behavior|
|Mapping Rules | Source metadata to target columns|
|SQL Standards | Databricks DDL generation requirements|
|DML Standards | Databricks DML (load) generation requirements|

## 4. Table Parameters

Only this section changes between survey sections.

| Parameter | Value |
| --- | --- |
| SECTION_NAME | <section_name> |
| SECTION_DESCRIPTION | RAND HRS Codebook – <Section Description> |
| CATALOG_NAME | dev_catalog |
| SCHEMA_NAME | slv_cdm_hrs |
| TABLE_NAME | hrs_<table_name> |
| FULL_TABLE_NAME | dev_catalog.slv_cdm_hrs.hrs_<table_name> |
| STORAGE_FORMAT | DELTA |
| TABLE_TYPE | Managed Table |
| LOAD_PATTERN | Insert Only |
| PRIMARY_KEY | hrs_<table_name>_id |

## 4a. Source Profile

**New in v2.0.** §4 alone was sufficient when every section pulled from the same
wide core file (`randhrs1992_2022v1`). Sections drawn from other RAND HRS file
families — e.g., the Leave-Behind Psychosocial Questionnaire — will differ on
every dimension below, and the DML pattern must be chosen accordingly.

| Property | Value | Notes |
| --- | --- | --- |
| SOURCE_TABLE_NAME | <catalog>.<schema>.<table> | Fully qualified bronze source |
| SOURCE_FILE_FAMILY | Core Longitudinal File \| Leave-Behind PSQ \| Tracker File \| Other | Determines whether this section shares a source table with siblings (Demographics, Health) or needs its own bronze ingestion |
| SOURCE_ROW_GRAIN | One row per respondent \| One row per respondent-wave | Core file = one row per respondent (wide, unpivoted here). Some source families are already long/tall per wave — if so, no UNPIVOT is needed at all, and this section's DML pattern should say so explicitly. |
| SOURCE_NAMING_CONVENTION | e.g. `R{wave}{VARNAME}` | State the actual regex/pattern. Leave-Behind variables typically do NOT follow the RAND-harmonized `Rw` prefix — confirm against the actual codebook before assuming the core pattern applies. |
| SAMPLE_COVERAGE | Full sample every wave \| Partial/rotating sample | Leave-Behind PSQ is administered to only half the sample per wave, alternating. This materially changes expected null density and should inform whether missing rows are treated as "not asked" vs. "data quality issue." |
| KNOWN_STRUCTURAL_DIFFERENCES | Free text | e.g. "Contains multi-item scale composites (e.g., CESD-8) requiring aggregation logic beyond a simple per-column TRY_CAST; contains reverse-scored items." |

## 5. Parent Tables

| Parent Table | Primary Key | Join Variable |
| --- | --- | --- |
| hrs_respondent | respondent_id | HHIDPN |
| hrs_wave | wave_id | wave_number |

## 6. Transformation Parameters

| RAND Variable Type | Parameter | Transformation |
| --- | --- | --- |
| CONT | CONT | TRY_CAST(<source_column> AS DECIMAL(10,2)) AS <target_column> |
| CATEG | CATEG | TRY_CAST(<source_column> AS <Databricks Type specified per-column in Section 12>) AS <target_column> |
| CHAR | CHAR | TRY_CAST(<source_column> AS STRING) AS <target_column> |

**Correction from v1.0:** the CATEG row previously hardcoded `AS INT`, which
contradicted Section 12's per-column Databricks Type (e.g., `tinyint` for
Health section flags). Section 12's declared Databricks Type is always
authoritative; this row exists to describe the transformation function
(`TRY_CAST`), not to fix the target type.

## 7. Dependencies

| Object | Requirement |
| --- | --- |
| Catalog | dev_catalog |
| Schema | dev_catalog.slv_cdm_hrs |
| Parent Table | dev_catalog.slv_cdm_hrs.hrs_respondent |
| Parent Table | dev_catalog.slv_cdm_hrs.hrs_wave |
| Source Table | <SOURCE_TABLE_NAME from §4a> |

## Parent Entities

| Parent Table | Primary Key |
| --- | --- |
| hrs_respondent | respondent_id |
| hrs_wave | wave_id |

## Child Entity

### Table

- hrs_<table_name>

## Relationships
| Parent | Child | Cardinality |
| --- | --- | --- |
| hrs_respondent | hrs_<table_name> | One to Many |
| hrs_wave | hrs_<table_name> | One to Many |

## Business Key
### Columns
- respondent_id
- wave_id

Together these columns uniquely identify one <section_name> observation.

## 9. Business Rules

| Rule | Description |
| --- | --- |
| Row Grain | One row per respondent per survey wave. **State the wave universe explicitly per section** — e.g. "Waves 1–16 (full study span)" or "Waves 8–16 only (section not administered before Wave 8)" — rather than assuming 1–16 by default. |
| Respondent FK | Required |
| Wave FK | Required |
| Duplicate Records | Not Allowed |
| Logical Business Key | respondent_id + wave_id |
| Unresolvable FK Handling | <Silently exclude via INNER JOIN \| Route to reject table \| Fail load> — **must be stated explicitly per section.** Default template behavior (used for Demographics/Health) is silent exclusion via INNER JOIN. Sections with partial sample coverage (§4a) should consider whether silent exclusion could mask genuine data-quality issues vs. expected non-participation. |

## 10. Physical Table Definition

| Property | Value |
| --- | --- |
| Storage Format | DELTA |
| Table Type | Managed Table |
| Table Comment | Stores RAND HRS <Section Description> observations |

## 11. Column Definitions

### Identity Column

| Column | Type | Nullable | Description |
| --- | --- | --- | --- |
| hrs_<table_name>_id | BIGINT | No | System-generated surrogate key |

Generated Always As Identity

### Foreign Keys

| Column | Type | Nullable | References |
| --- | --- | --- | --- |
| respondent_id | BIGINT | No | hrs_respondent.respondent_id |
| wave_id | BIGINT | No | hrs_wave.wave_id |

### Audit Columns

| Column | Type | Nullable | Description |
| --- | --- | --- | --- |
| create_date | DATE | No | Record creation date |
| update_date | DATE | No | Last update date |
| active | BOOLEAN | No | Active indicator |

### Identifier Columns

|Column | Type | Nullable | Description |
| --- | --- | --- | --- |
|hhidpn | int | Yes | Household Respondent Identifier |
|wave_number | string | No | Wave Number |

### Business Columns

The following columns are generated from the Source-to-Target Mapping Matrix (§12).

## 12. Source-to-Target Mapping Matrix

This section is replaced for each survey section.

State explicitly (do not leave implicit, per Gap #1 from the v1.0 review):
- The Wave column's values are strings and must be handled as STRING type
  throughout DDL and DML, unless a future section overrides this in §4a.
- Whether target columns in this section are wave-varying, wave-invariant, or
  a mix — cross-reference §12a below rather than stating this only in prose.

### Target Column: <target_column_name>
| Wave | Source Variable | Variable Label | RAND Type | Target Column | Databricks Type | Transformation | Nullable |
| --- | --- | --- | --- | --- | --- | --- | --- |
| 1 | <R1VARNAME> | <label> | <CONT/CATEG/CHAR> | <target_column_name> | <type> | <CONT/CATEG/CHAR> | yes/no |
| ... | ... | ... | ... | ... | ... | ... | ... |
| 16 | <R16VARNAME> | <label> | <CONT/CATEG/CHAR> | <target_column_name> | <type> | <CONT/CATEG/CHAR> | yes/no |

*(Repeat this block per target column. For wave-invariant columns, include only
a single row with no wave-number prefix on the source variable — see §12a.)*

## 12a. Column Variance Declaration

**New in v2.0.** Every business column must be explicitly classified here. This
is the single source of truth that determines which DML pattern (§13a) applies
to each column — do not infer variance from the mapping matrix's row count alone.

| Target Column | Variance | Source Pattern | Notes |
| --- | --- | --- | --- |
| <target_column_name> | Wave-varying \| Wave-invariant | `R{w}VARNAME` \| `VARNAME` (no wave prefix) | Wave-invariant columns are computed once and joined/replicated across all wave rows; wave-varying columns are unpivoted per wave. |

*Example (Demographics): `raracem`, `rahispan`, `raedyrs`, `rarelig`, `ravetrn`
are wave-invariant; `agey_e`, `cenreg`, `mstat` are wave-varying.*
*Example (Health): all 10 business columns are wave-varying; none are
wave-invariant.*

## 13. SQL Generation Requirements (DDL)

| Requirement | Required |
| --- | --- |
| DROP TABLE IF EXISTS | Yes |
| CREATE TABLE | Yes |
| USING DELTA | Yes |
| GENERATED ALWAYS AS IDENTITY | Yes |
| Table Comment | Yes |
| Column Comments | Yes |
| Explicit Column Definitions | Yes |
| Uppercase SQL Keywords | Yes |
| Consistent Indentation | Yes |

## 13a. DML Generation Requirements

**New in v2.0.** These rules were previously undocumented and were carried
forward only by convention across the Demographics and Health DML scripts.
Stating them explicitly ensures Section L (and beyond) are generated
consistently without relying on prior conversational context.

| Requirement | Rule |
| --- | --- |
| Unpivot Method | If SOURCE_ROW_GRAIN (§4a) is "one row per respondent" and any wave-varying columns exist, use a single **multi-value `UNPIVOT ... INCLUDE NULLS`** statement covering all wave-varying columns together. Do not use per-column UNPIVOT (requires re-joining) or UNION ALL-per-wave (requires N scans) — both are less efficient and were superseded during Health section development. |
| Wave-Invariant Columns | Computed once in a `source_base`-equivalent CTE, then joined (or cross-joined via the unpivot's natural row replication) onto every wave row. Do not unpivot wave-invariant columns. |
| NULL Handling | No `WHERE ... IS NOT NULL` filtering on business columns. Any single business column may legitimately be NULL for a given respondent/wave without invalidating the row. Use `INCLUDE NULLS` in the UNPIVOT clause. |
| FK Join Type | `INNER JOIN` against `hrs_respondent` and `hrs_wave`, per the Unresolvable FK Handling rule declared in §9 for this section. |
| Type Casting | All casts use `TRY_CAST`, never `CAST`, per §6. Cast target types match §12's declared Databricks Type exactly. |
| wave_number Join Key | Cast/aliased consistently as STRING on both sides of the join — confirm §4a hasn't overridden this for a section with an integer-typed source wave indicator. |
| Audit Column Population | `create_date` and `update_date` both populated via `CURRENT_DATE()` on initial load. `active` hardcoded to `TRUE`. (Insert-Only load pattern per §4 — no update/soft-delete logic in scope.) |
| CTE Naming Convention | `source_base` (raw wide columns) → `wave_unpivoted_raw` (post-UNPIVOT, pre-cast) → `wave_unpivoted` (post-TRY_CAST) → final `SELECT` with FK joins. Keep this naming consistent across sections for readability. |
| Missing Source Columns for Some Waves | If §12 shows a target column's wave coverage starting after Wave 1 (or ending before Wave 16), the source table has **no column at all** for the missing waves (not merely NULL values) — referencing `R{n}VARNAME` for a wave where it doesn't exist raises a column-not-found error inside `UNPIVOT`. If the section's declared Row Grain (§9) still requires rows for the full 1–16 wave universe, build the DML as a `UNION ALL` of: (a) a real `UNPIVOT` over the waves where source columns exist, and (b) a synthetic "scaffold" — literal wave-number values cross-joined against every `HHIDPN`, with all business columns hardcoded `NULL` — for the waves where they don't. Do not attempt to include non-existent columns in the UNPIVOT tuple list under any circumstance. |

## 14. Constraint Requirements

### Primary Key Constraint
- CONSTRAINT pk_<table_name> PRIMARY KEY (<PRIMARY_KEY>)

### Foreign Key Constraints
CONSTRAINT fk_<table_name>_respondent
    FOREIGN KEY (respondent_id)
    REFERENCES dev_catalog.slv_cdm_hrs.hrs_respondent (respondent_id),

CONSTRAINT fk_<table_name>_wave
    FOREIGN KEY (wave_id)
    REFERENCES dev_catalog.slv_cdm_hrs.hrs_wave (wave_id)

### Business Key Uniqueness Constraint

Databricks Runtime 15.x supports: PRIMARY KEY, FOREIGN KEY, UNIQUE, CHECK,
NOT NULL, GENERATED ALWAYS AS IDENTITY.

| Constraint Type | Required | Notes |
| --- | --- | --- |
| PRIMARY KEY | ✓ | Enforced at write time |
| FOREIGN KEY | ✓ | Enforced at write time |
| UNIQUE | ✓ | Required for business key |
| NOT NULL | ✓ | Required for audit + FK columns |
| CHECK | Optional | Evaluate per section — e.g. useful for scale-bounded composite scores in psychosocial sections |
| IDENTITY | ✓ | Required for surrogate key |

CONSTRAINT uq_<table_name>_respondent_wave UNIQUE (respondent_id, wave_id)

### Do not generate (DDL or DML)

Unsupported Objects:
- UPDATE, DELETE, MERGE
- Views, Indexes, Partitions, ZORDER, OPTIMIZE

(INSERT is required for DML deliverables — the v1.0 exclusion list incorrectly
grouped INSERT with UPDATE/DELETE/MERGE. Corrected here: INSERT is in scope;
UPDATE/DELETE/MERGE remain out of scope under the Insert-Only load pattern.)

## Column Constraint Rules

### Identity Column
hrs_<table_name>_id BIGINT GENERATED ALWAYS AS IDENTITY
    CONSTRAINT pk_<table_name> PRIMARY KEY

### Foreign Keys
respondent_id BIGINT NOT NULL
    CONSTRAINT fk_<table_name>_respondent
        FOREIGN KEY REFERENCES dev_catalog.slv_cdm_hrs.hrs_respondent (respondent_id),

wave_id BIGINT NOT NULL
    CONSTRAINT fk_<table_name>_wave
        FOREIGN KEY REFERENCES dev_catalog.slv_cdm_hrs.hrs_wave (wave_id)

### Business Key
CONSTRAINT uq_<table_name>_respondent_wave UNIQUE (respondent_id, wave_id)

## 15. Validation Requirements

| Validation | Required |
| --- | --- |
| Table Exists | ✓ |
| Correct Schema | ✓ |
| Delta Format | ✓ |
| Managed Table | ✓ |
| Identity Column Exists | ✓ |
| Respondent FK Exists | ✓ |
| Wave FK Exists | ✓ |
| Audit Columns Exist | ✓ |
| respondent_id exists in hrs_respondent | ✓ |
| wave_id exists in hrs_wave | ✓ |
| No duplicate respondent_id + wave_id | ✓ |
| Rows loaded > 0 | ✓ |
| wave_number values within expected wave range | ✓ |
| Business column domain checks (per §12a) | Recommended — e.g. binary flags in {0,1}, categorical scales within documented bounds |

Example Constraint Block (Template):
```
CONSTRAINT pk_<table_name> PRIMARY KEY (<PRIMARY_KEY>),
CONSTRAINT fk_<table_name>_respondent FOREIGN KEY (respondent_id)
    REFERENCES dev_catalog.slv_cdm_hrs.hrs_respondent (respondent_id),
CONSTRAINT fk_<table_name>_wave FOREIGN KEY (wave_id)
    REFERENCES dev_catalog.slv_cdm_hrs.hrs_wave (wave_id),
CONSTRAINT uq_<table_name>_respondent_wave UNIQUE (respondent_id, wave_id)
```

## 16. Deliverables

|Item | Value|
|---|---|
|DDL File| /sql/ddl/create_hrs_<table_name>.sql|
|DML File| /sql/dml/insert_hrs_<table_name>.sql|
|Validation File| /sql/validation/validate_hrs_<table_name>.sql|
|Output| SQL Only|

## 17. Worked Example Reference

**New in v2.0.** Rather than re-deriving DML conventions from scratch for each
new section, use the Health section as the reference implementation for any
section whose Source Profile (§4a) matches Core Longitudinal File / one-row-
per-respondent / full-sample-coverage:

- DDL: `create_hrs_health.sql`
- DML: `insert_hrs_health.sql` (multi-value UNPIVOT pattern)
- Validation: `validate_hrs_health.sql`

For sections with a different Source Profile (e.g., Leave-Behind PSQ — separate
source file, partial sample, non-`Rw`-prefixed variable names, possible
multi-item scale composites), §13a's DML rules should be revisited before
assuming the Health pattern applies unmodified. In particular:
- If SOURCE_ROW_GRAIN is already "one row per respondent-wave," no UNPIVOT is
  needed at all — the DML becomes a straight `SELECT ... TRY_CAST(...) FROM
  source` with FK joins.
- If the section includes multi-item scale composites, §12 should document the
  aggregation logic (e.g., mean-of-items, reverse-scoring) per target column,
  and §13a should specify where that aggregation happens (in the unpivot CTE,
  or in a subsequent CTE after unpivoting individual items).